In [28]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas

!uv pip uninstall transformers tokenizers accelerate -q

!uv pip install "transformers==4.56.0" "protobuf==5.29.3" -q
!uv pip install torch datasets -q
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
# !uv pip install -r requirements.txt
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

Using Python 3.12.12 environment at: /usr
Resolved 1 package in 41ms                                           
Audited 1 package in 0.19ms
Using Python 3.12.12 environment at: /usr
Audited 3 packages in 131ms
Using Python 3.12.12 environment at: /usr
Uninstalled 1 package in 156ms
 - pandas==3.0.2
Using Python 3.12.12 environment at: /usr
Audited 2 packages in 131ms
Using Python 3.12.12 environment at: /usr
Resolved 44 packages in 21ms                                         
Installed 1 package in 5ms                                  
 + accelerate==1.13.0
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 146ms                                          
Prepared 1 package in 359ms                                              
Uninstalled 1 package in 35ms
Installed 1 package in 22ms                                 
 ~ numpy==2.0.0


In [42]:
import os
import re
import logging
import sys
import torch
import torch.nn as nn
import wandb
from typing import Dict, Tuple
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import subprocess

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

# Log environment information
logger.info("\n" + "=" * 80)
logger.info("ENVIRONMENT CONFIGURATION")
logger.info("=" * 80)
logger.info(f"Python version: {sys.version}")
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"CUDA version: {torch.version.cuda}")
    logger.info(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        logger.info(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    logger.info("Running on CPU")

try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        logger.info(f"NVIDIA GPU: {result.stdout.strip()}")
except Exception:
    pass
logger.info("=" * 80 + "\n")
wandb.login()

2026-04-15 09:45:42,239 - INFO - 
2026-04-15 09:45:42,239 - INFO - 
04/15/2026 09:45:42 - INFO - __main__ - 
2026-04-15 09:45:42,243 - INFO - ENVIRONMENT CONFIGURATION
2026-04-15 09:45:42,243 - INFO - ENVIRONMENT CONFIGURATION
04/15/2026 09:45:42 - INFO - __main__ - ENVIRONMENT CONFIGURATION
2026-04-15 09:45:42,247 - INFO - ================================================================================
2026-04-15 09:45:42,247 - INFO - ================================================================================
04/15/2026 09:45:42 - INFO - __main__ - ================================================================================
2026-04-15 09:45:42,252 - INFO - Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
2026-04-15 09:45:42,252 - INFO - Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
04/15/2026 09:45:42 - INFO - __main__ - Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
2026-04-15 09:45:42,255 - INFO - PyTorch version: 2

False

In [29]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

def preprocess_code(code_str: str) -> str:
    """Normalize code string."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)
    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()


def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)


def load_datasets(tokenizer, max_length: int = 512) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets."""
    logger.info("Loading datasets from Hugging Face Hub...")

    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

    logger.info(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
    )

    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
    )

    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    return train_dataset, val_dataset

# ============================================================================
# 1.5 METRICS: Evaluation metric function for Trainer
# ============================================================================

def compute_metrics_fn(eval_pred):
    """Compute precision, recall, F1, and accuracy for evaluation."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

In [30]:
# ============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# ============================================================================

# Note: The Trainer class from transformers handles:
# - Training loop with gradient accumulation
# - Validation every N steps (eval_steps=200)
# - Best model selection based on macro_f1
# - Automatic W&B logging if enabled
# - Checkpointing and model saving

In [31]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(model_name: str, num_labels: int = 2):
    """Load pretrained model and tokenizer from Hugging Face."""
    logger.info(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    logger.info(f"Loading model from: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )

    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")

    return model, tokenizer


def freeze_base_model(model):
    """
    Freeze all base model parameters, keeping only classifier head trainable.

    For sequence classification models (e.g., CodeBERT), this freezes the
    RoBERTa/encoder layers and unfreezes the classification head.

    Args:
        model: The pretrained model to freeze
    """
    # Freeze all encoder/base model parameters
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False

    # Ensure classifier head is trainable
    for name, param in model.named_parameters():
        if "classifier" in name or "cls" in name.lower():
            param.requires_grad = True

    logger.info("Base model frozen - only classifier head is trainable")


In [46]:
# =============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# =============================================================================

import os
import logging
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple

import torch
from torch import nn
from torch.nn import functional as F
from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# ----------------------------------------------------------------------
# Configuration dataclass
# ----------------------------------------------------------------------
@dataclass
class TrainConfig:
    model_name: str = "microsoft/codebert-base"
    output_dir: str = "./taskA-codebert-base"
    num_epochs: int = 3
    batch_size: int = 32
    learning_rate: float = 2e-5
    max_length: int = 512
    num_labels: int = 2
    use_wandb: bool = False
    freeze_base: bool = False
    loss_type: str = "ce"               # "ce", "focal", "infonce", "r-drop"
    focal_alpha: float = 1.0
    focal_gamma: float = 2.0
    r_drop_alpha: float = 4.0
    infonce_temperature: float = 0.07
    infonce_weight: float = 0.5
    seed: int = 42
    resume_from_checkpoint: str = None
    # Internally populated
    device: torch.device = field(init=False, default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))


# ----------------------------------------------------------------------
# Logging utilities
# ----------------------------------------------------------------------
def setup_logger(output_dir: str) -> logging.Logger:
    """Create a logger that writes to both console and a file."""
    logger = logging.getLogger("train_pipeline")
    logger.setLevel(logging.INFO)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(ch)

    # File handler (overwrite each run)
    log_path = os.path.join(output_dir, "training.log")
    fh = logging.FileHandler(log_path, mode="w")
    fh.setLevel(logging.INFO)
    fh.setFormatter(
        logging.Formatter("%(asctime)s - %(levelname)s - %(name)s - %(message)s")
    )
    logger.addHandler(fh)

    logger.info(f"Logging to {log_path}")
    return logger


# ----------------------------------------------------------------------
# Model / tokenizer loading
# ----------------------------------------------------------------------
def load_model_and_tokenizer(
    model_name: str,
    num_labels: int,
    device: torch.device,
    logger: logging.Logger,
) -> Tuple[nn.Module, "PreTrainedTokenizer"]:
    """Load a HuggingFace model + tokenizer and move the model to `device`."""
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    logger.info(f"Loading model & tokenizer for '{model_name}'")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
    )
    model.to(device)
    logger.info(f"Model placed on {device}")
    return model, tokenizer


# ----------------------------------------------------------------------
# Optional weight freezing
# ----------------------------------------------------------------------
def freeze_base_model(model: nn.Module, logger: logging.Logger) -> None:
    """Freeze all parameters except the classification head."""
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False
    logger.info("Base model weights frozen – only classifier head will be trained.")


# ----------------------------------------------------------------------
# Loss function factories
# ----------------------------------------------------------------------
def get_focal_loss(alpha: float, gamma: float) -> Callable:
    def focal_loss(outputs, labels, **_):
        logits = outputs.logits
        ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        loss = alpha * (1 - pt) ** gamma * ce
        return loss.mean()
    return focal_loss


def get_infonce_loss(temperature: float, weight: float) -> Callable:
    def infonce_loss(outputs, labels, **_):
        # Assuming `outputs.hidden_states[-1]` is the pooled representation
        reps = outputs.hidden_states[-1]  # (batch, hidden)
        reps = F.normalize(reps, dim=-1)
        sim_matrix = torch.mm(reps, reps.t()) / temperature
        # Positive pairs are diagonal offsets; negatives are all other entries
        labels = torch.arange(reps.size(0), device=reps.device)
        loss = F.cross_entropy(sim_matrix, labels)
        return weight * loss
    return infonce_loss


# ----------------------------------------------------------------------
# Helper: log model architecture (console + file)
# ----------------------------------------------------------------------
def log_model_architecture(model: nn.Module, tokenizer, logger: logging.Logger) -> None:
    """
    Print a concise but complete description of the model and tokenizer.
    The logger writes to both stdout and the training log file.
    """
    logger.info("===== Model Architecture =====")
    # `repr(model)` gives the full hierarchical description used by HF models.
    logger.info("\n" + model.__repr__())

    # Tokenizer summary – useful for debugging vocab size, special tokens, etc.
    logger.info("===== Tokenizer Summary =====")
    logger.info(
        f"Vocab size: {len(tokenizer)} | "
        f"Special tokens: {tokenizer.all_special_tokens}"
    )
    logger.info("===== End of Architecture Log =====")


# ----------------------------------------------------------------------
# Custom Trainer with unified loss handling
# ----------------------------------------------------------------------
class CustomTrainer(Trainer):
    """Trainer that supports CE, Focal, InfoNCE, and R‑Drop losses."""

    def __init__(
        self,
        *args,
        loss_type: str = "ce",
        r_drop_alpha: float = 4.0,
        compute_loss_fn: Optional[Callable] = None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.loss_type = loss_type
        self.r_drop_alpha = r_drop_alpha
        self.compute_loss_fn = compute_loss_fn

    def compute_loss(self, model, inputs, num_items_in_batch = 64, return_outputs: bool = False):
        """Override Trainer.compute_loss to inject custom loss logic."""
        labels = inputs.pop("labels", None)

        # ----- R‑Drop (dual forward) -----
        if self.loss_type == "r-drop" and self.training and labels is not None:
            out1 = model(**inputs)
            out2 = model(**inputs)

            ce = (F.cross_entropy(out1.logits, labels) + F.cross_entropy(out2.logits, labels)) / 2

            p = F.log_softmax(out1.logits, dim=-1)
            q = F.log_softmax(out2.logits, dim=-1)

            kl = (
                F.kl_div(p, F.softmax(out2.logits, dim=-1), reduction="batchmean")
                + F.kl_div(q, F.softmax(out1.logits, dim=-1), reduction="batchmean")
            ) / 2

            loss = ce + self.r_drop_alpha * kl
            return (loss, out1) if return_outputs else loss

        # ----- Standard forward -----
        outputs = model(**inputs)

        if labels is None:
            loss = outputs.loss if hasattr(outputs, "loss") else outputs[0]
        else:
            if self.compute_loss_fn and self.loss_type != "r-drop":
                loss = self.compute_loss_fn(outputs, labels)
            else:
                loss = F.cross_entropy(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss


# ----------------------------------------------------------------------
# Main training pipeline
# ----------------------------------------------------------------------
def train_pipeline(cfg: TrainConfig) -> Tuple[Trainer, nn.Module, "PreTrainedTokenizer"]:
    """Run the full training loop and return the trainer, model and tokenizer."""
    os.makedirs(cfg.output_dir, exist_ok=True)
    logger = setup_logger(cfg.output_dir)

    # Load model & tokenizer
    model, tokenizer = load_model_and_tokenizer(
        cfg.model_name, cfg.num_labels, cfg.device, logger
    )

    # ------------------------------------------------------------------
    # NEW: Log architecture before any training starts
    # ------------------------------------------------------------------
    log_model_architecture(model, tokenizer, logger)

    # Optional hidden‑state exposure for InfoNCE
    if cfg.loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states for InfoNCE loss.")

    # Freeze base if requested
    if cfg.freeze_base:
        freeze_base_model(model, logger)

    # Load datasets (user‑provided helpers)
    train_dataset, val_dataset = load_datasets(tokenizer, cfg.max_length)

    # WandB optional setup
    if cfg.use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "online"
        except Exception as e:
            logger.warning(f"W&B import failed ({e}); proceeding without it.")
            cfg.use_wandb = False

    # Compute steps for scheduler
    steps_per_epoch = max(1, len(train_dataset) // cfg.batch_size)
    total_steps = cfg.num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))

    # TrainingArguments
    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        learning_rate=cfg.learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        logging_strategy="steps",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to=["wandb"] if cfg.use_wandb else [],
        run_name="codebert-base-task-a" if cfg.use_wandb else None,
        gradient_checkpointing=True,
        dataloader_pin_memory=torch.cuda.is_available(),
        seed=cfg.seed,
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    # Choose loss implementation
    compute_loss_fn = None
    if cfg.loss_type == "focal":
        compute_loss_fn = get_focal_loss(cfg.focal_alpha, cfg.focal_gamma)
    elif cfg.loss_type == "infonce":
        compute_loss_fn = get_infonce_loss(cfg.infonce_temperature, cfg.infonce_weight)

    # Instantiate trainer
    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        loss_type=cfg.loss_type,
        r_drop_alpha=cfg.r_drop_alpha,
        compute_loss_fn=compute_loss_fn,
    )

    logger.info("=== Starting training ===")
    trainer.train(resume_from_checkpoint=cfg.resume_from_checkpoint)
    logger.info("🎯 Training complete!")
    logger.info(f"Best model saved to {os.path.join(cfg.output_dir, 'best_model')}")

    return trainer, model, tokenizer


# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    cfg = TrainConfig(
        model_name="microsoft/codebert-base",
        output_dir="./taskA-codebert-base",
        num_epochs=3,
        batch_size=32,
        learning_rate=2e-5,
        max_length=512,
        use_wandb=True,
        freeze_base=True,
        loss_type="ce",
        focal_alpha=1.0,
        focal_gamma=2.0,
        r_drop_alpha=4.0,
        infonce_temperature=0.07,
        infonce_weight=0.5,
        resume_from_checkpoint="./taskA-codebert-base/checkpoint-1000"
    )
    trainer, model, tokenizer = train_pipeline(cfg)

2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,900 - INFO - Logging to ./taskA-codebert-base/training.log
04/15/2026 10:25:16 - INFO - train_pipeline - Logging to ./taskA-codebert-base/training.log
2026-04-15 10:25:16,911 - INFO - Loading model & tokenizer for 'microsoft/codebert-base'
2026-04-15 10:25:16,911 - INFO - Loading model & tokenizer for 'microsoft/codebert-base'
2026-04-15 10:25:16,911 - INFO - Loading model & tokenizer for 'microsoft/codebert-base'
2026-04-15 10:25:16,911 - INFO - Loading model & tokenizer for 'microsoft/codebert-base'

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,Macro F1
1200,0.638200,0.622675,0.700030,0.699372,0.699266,0.699314


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


KeyboardInterrupt: 

In [49]:
import os
import logging
from pathlib import Path
from typing import Tuple, Dict, Any

import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# New imports for metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ----------------------------------------------------------------------
# 1. SETUP LOGGER
# ----------------------------------------------------------------------
def setup_logger(output_dir: str) -> logging.Logger:
    logger = logging.getLogger("inference")
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        ch = logging.StreamHandler()
        ch.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
        logger.addHandler(ch)

        os.makedirs(output_dir, exist_ok=True)
        log_path = Path(output_dir) / "inference.log"
        fh = logging.FileHandler(log_path, mode="w")
        fh.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
        logger.addHandler(fh)
    return logger

def log_model_architecture(model: nn.Module, tokenizer, logger: logging.Logger) -> None:
    """
    Print a concise but complete description of the model and tokenizer.
    The logger writes to both stdout and the training log file.
    """
    logger.info("===== Model Architecture =====")
    # `repr(model)` gives the full hierarchical description used by HF models.
    logger.info("\n" + model.__repr__())

    # Tokenizer summary – useful for debugging vocab size, special tokens, etc.
    logger.info("===== Tokenizer Summary =====")
    logger.info(
        f"Vocab size: {len(tokenizer)} | "
        f"Special tokens: {tokenizer.all_special_tokens}"
    )
    logger.info("===== End of Architecture Log =====")

# ----------------------------------------------------------------------
# 2. MAIN INFERENCE PIPELINE WITH METRICS
# ----------------------------------------------------------------------
def run_standalone_inference(
    checkpoint_path: str,
    output_dir: str = "./",
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
    dataset_name: str = "DaniilOr/SemEval-2026-Task13",
    dataset_config: str = "A",
    split: str = "test",
):
    logger = setup_logger(output_dir)
    logger.info(f"Loading model and tokenizer from: {checkpoint_path}")

    # --- Load Model & Tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
    log_model_architecture(model, tokenizer, logger)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # --- Load Dataset ---
    logger.info(f"Loading dataset: {dataset_name} ({dataset_config})")
    test_ds = load_dataset(dataset_name, dataset_config, split=split)

    # --- Tokenization ---
    def tokenize_fn(examples):
        return tokenizer(
            examples["code"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    logger.info("Tokenizing dataset...")
    tokenized_ds = test_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=[c for c in test_ds.column_names if c not in ["input_ids", "attention_mask", "id", "label"]],
        desc="Tokenizing"
    )
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    # --- DataLoader ---
    test_loader = torch.utils.data.DataLoader(
        tokenized_ds,
        batch_size=batch_size,
        shuffle=False,
    )

    # --- Inference Loop ---
    logger.info(f"Running inference on {len(test_ds)} examples...")
    all_logits = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    pred_labels = logits.argmax(axis=-1)

    # ------------------------------------------------------------------
    # 3. METRIC CALCULATION
    # ------------------------------------------------------------------
    if "label" in test_ds.column_names:
        logger.info("Calculating classification metrics...")
        true_labels = np.array(test_ds["label"])
        
        # Calculate Accuracy
        acc = accuracy_score(true_labels, pred_labels)
        
        # Calculate Precision, Recall, F1 (weighted handles class imbalance)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average='weighted'
        )
        
        logger.info("-" * 30)
        logger.info(f"METRICS FOR SPLIT: {split}")
        logger.info(f"Accuracy:  {acc:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        logger.info("-" * 30)
        
        # Optional: Print Confusion Matrix
        cm = confusion_matrix(true_labels, pred_labels)
        logger.info(f"Confusion Matrix:\n{cm}")
    else:
        logger.warning("No 'label' column found in dataset. Skipping metric calculation.")

    # --- Save Results ---
    csv_path = Path(output_dir) / output_csv
    ids = test_ds["id"] if "id" in test_ds.column_names else list(range(len(pred_labels)))

    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("id,label\n")
        for idx, label in zip(ids, pred_labels):
            f.write(f"{idx},{label}\n")

    logger.info(f"✅ Predictions saved to {csv_path}")

# ----------------------------------------------------------------------
# 4. EXECUTION
# ----------------------------------------------------------------------
if __name__ == "__main__":
    CHECKPOINT = "./taskA-codebert-base/checkpoint-1000" 
    
    if os.path.exists(CHECKPOINT):
        run_standalone_inference(
            checkpoint_path=CHECKPOINT,
            output_dir="./outputs",
            output_csv="submission.csv",
            batch_size=64
        )

2026-04-15 11:04:58,299 - INFO - Loading model and tokenizer from: ./taskA-codebert-base/checkpoint-1000
04/15/2026 11:04:58 - INFO - inference - Loading model and tokenizer from: ./taskA-codebert-base/checkpoint-1000
2026-04-15 11:04:58,470 - INFO - ===== Model Architecture =====
04/15/2026 11:04:58 - INFO - inference - ===== Model Architecture =====
2026-04-15 11:04:58,474 - INFO - 
RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_

Tokenizing:   0%|          | 0/1000 [00:00<?, ? examples/s]

2026-04-15 11:04:59,823 - INFO - Running inference on 1000 examples...
04/15/2026 11:04:59 - INFO - inference - Running inference on 1000 examples...


Predicting:   0%|          | 0/16 [00:00<?, ?it/s]

2026-04-15 11:05:34,869 - INFO - Calculating classification metrics...
04/15/2026 11:05:34 - INFO - inference - Calculating classification metrics...
2026-04-15 11:05:34,889 - INFO - ------------------------------
04/15/2026 11:05:34 - INFO - inference - ------------------------------
2026-04-15 11:05:34,891 - INFO - METRICS FOR SPLIT: test
04/15/2026 11:05:34 - INFO - inference - METRICS FOR SPLIT: test
2026-04-15 11:05:34,894 - INFO - Accuracy:  0.3860
04/15/2026 11:05:34 - INFO - inference - Accuracy:  0.3860
2026-04-15 11:05:34,896 - INFO - Precision: 0.7077
04/15/2026 11:05:34 - INFO - inference - Precision: 0.7077
2026-04-15 11:05:34,899 - INFO - Recall:    0.3860
04/15/2026 11:05:34 - INFO - inference - Recall:    0.3860
2026-04-15 11:05:34,902 - INFO - F1-Score:  0.3913
04/15/2026 11:05:34 - INFO - inference - F1-Score:  0.3913
2026-04-15 11:05:34,905 - INFO - ------------------------------
04/15/2026 11:05:34 - INFO - inference - ------------------------------
2026-04-15 11:05

In [47]:
from huggingface_hub import HfApi

api = HfApi()

# Replace these with your details
repo_id = "dzungpham/SLA-SemEval-challenge"
local_folder_path = "./taskA-codebert-base"

# Create the repo first (optional, will skip if it exists)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload the entire folder
api.upload_folder(
    folder_path=local_folder_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="upload model weights after 1200 training steps with frozen pretrained weights"
)

print(f"✅ Upload complete! View at: https://huggingface.co/{repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Upload complete! View at: https://huggingface.co/dzungpham/SLA-SemEval-challenge
